# Stop-Codon-Free GA Optimization

This notebook tests the modified Genetic Algorithm that **guarantees all evolutionary paths are stop-codon free** at every intermediate step.

## Key Changes from Standard GA:
1. **Population Initialization**: Random permutations are regenerated until stop-codon free
2. **Crossover**: Retry with different crossover points if child has stop codons; fallback to parent
3. **Mutation**: Resample swap positions if mutation creates stop codons; return original if no valid swap

## Workflow:
1. Generate functional overlapping sequences using Monte Carlo
2. Run GA optimization with stop-codon-free guarantee
3. **Validate** all paths are actually stop-codon free
4. Analyze results and compare with baseline

## 1. Setup and Imports

In [4]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import sys
import os
import time
import platform
from scipy import stats
import matplotlib.cm as cm
from matplotlib.colors import Normalize

# Multiprocessing imports
from multiprocessing import Pool, cpu_count
from functools import partial

# Progress bar
from tqdm.auto import tqdm

# Add parent directory to path for imports
sys.path.insert(0, os.path.dirname(os.getcwd()))
import overlappingGenes as og

# Import from modified ga_worker.py
from ga_worker import (
    GeneticPathFinder,
    is_path_stop_codon_free,
    seq_to_array,
    get_mutations_arrays,
    hamming_distance,
    find_closest_pair,
    get_path_energies,
    evaluate_path_fitness_numba
)

# Plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('viridis')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

# System info
print("=" * 60)
print("SYSTEM INFORMATION")
print("=" * 60)
print(f"Platform:        {platform.system()} {platform.release()}")
print(f"Python version:  {platform.python_version()}")

N_WORKERS = cpu_count()
print(f"\nCPU Cores Detected: {N_WORKERS}")
print("=" * 60)
print("\nImports complete.")

SYSTEM INFORMATION
Platform:        Windows 11
Python version:  3.13.9

CPU Cores Detected: 12

Imports complete.


## 2. Configuration

In [5]:
# =============================================================================
# EXPERIMENT CONFIGURATION
# =============================================================================

# Protein families
PROTEIN_1 = 'PF00004'
PROTEIN_2 = 'PF00041'

# Overlaps to test (covering all three reading frames)
OVERLAPS = [12]  # Frames 0, 1, 2

# Sequence generation
N_SEQUENCES_PER_OVERLAP = 1  # Number of functional sequences per overlap
MC_ITERATIONS = 200_000       # MC iterations per sequence
MC_TEMP_1 = 0.818
MC_TEMP_2 = 0.955

# GA parameters (with stop-codon-free guarantee)
GA_POPULATION = 75
GA_GENERATIONS = 150
GA_MUTATION_RATE = 0.15

# Convergence tracking - save path every N generations
TRACK_EVERY_N_GENERATIONS = 10

# Stop-codon-free retry limits
MAX_INIT_RETRIES = 1000
MAX_MUTATION_RETRIES = 10
MAX_CROSSOVER_RETRIES = 10

# Number of trials per overlap (sequence pairs to analyze)
N_TRIALS_PER_OVERLAP = 5

# Normalization
Z_SCORE = True
DIST_LABEL = "Z-Score Distance" if Z_SCORE else "Distance from Natural"

print("=" * 60)
print("STOP-CODON-FREE GA CONFIGURATION")
print("=" * 60)
print(f"Proteins: {PROTEIN_1} + {PROTEIN_2}")
print(f"Overlaps: {OVERLAPS}")
print(f"Sequences per overlap: {N_SEQUENCES_PER_OVERLAP}")
print(f"Trials per overlap: {N_TRIALS_PER_OVERLAP}")
print(f"GA: pop={GA_POPULATION}, gen={GA_GENERATIONS}")
print(f"Track convergence every: {TRACK_EVERY_N_GENERATIONS} generations")
print(f"Retry limits: init={MAX_INIT_RETRIES}, mut={MAX_MUTATION_RETRIES}, cross={MAX_CROSSOVER_RETRIES}")
print(f"Z-Score normalization: {Z_SCORE}")
print(f"CPU Workers: {N_WORKERS}")
print("=" * 60)

STOP-CODON-FREE GA CONFIGURATION
Proteins: PF00004 + PF00041
Overlaps: [12]
Sequences per overlap: 1
Trials per overlap: 5
GA: pop=75, gen=150
Track convergence every: 10 generations
Retry limits: init=1000, mut=10, cross=10
Z-Score normalization: True
CPU Workers: 12


## 3. Load DCA Parameters

In [6]:
# Load DCA parameters for both proteins
print(f"Loading DCA parameters for {PROTEIN_1}...")
Jvec_1, hvec_1 = og.extract_params(f'{PROTEIN_1}/{PROTEIN_1}_params.dat')
nat_energies_1 = og.load_natural_energies(f'{PROTEIN_1}/{PROTEIN_1}_naturalenergies.txt')
mean_e1, std_e1 = np.mean(nat_energies_1), np.std(nat_energies_1)
print(f"  Length: {len(hvec_1)//21} AA, Mean energy: {mean_e1:.2f} +/- {std_e1:.2f}")

print(f"\nLoading DCA parameters for {PROTEIN_2}...")
Jvec_2, hvec_2 = og.extract_params(f'{PROTEIN_2}/{PROTEIN_2}_params.dat')
nat_energies_2 = og.load_natural_energies(f'{PROTEIN_2}/{PROTEIN_2}_naturalenergies.txt')
mean_e2, std_e2 = np.mean(nat_energies_2), np.std(nat_energies_2)
print(f"  Length: {len(hvec_2)//21} AA, Mean energy: {mean_e2:.2f} +/- {std_e2:.2f}")

# Store DCA params as tuples
DCA_params_1 = (Jvec_1, hvec_1)
DCA_params_2 = (Jvec_2, hvec_2)

# Protein lengths
prot1_len = len(hvec_1) // 21
prot2_len = len(hvec_2) // 21
len_aa_1 = prot1_len + 1  # +1 for stop codon
len_aa_2 = prot2_len + 1

print(f"\nProtein 1 length: {prot1_len} AA")
print(f"Protein 2 length: {prot2_len} AA")

Loading DCA parameters for PF00004...
  Length: 110 AA, Mean energy: 145.88 +/- 38.25

Loading DCA parameters for PF00041...
  Length: 74 AA, Mean energy: 120.66 +/- 17.64

Protein 1 length: 110 AA
Protein 2 length: 74 AA


## 4. JIT Warmup

In [7]:
# Warm up Numba JIT compilation
print("Warming up JIT compilation...")

# Generate a test sequence
test_overlap = 12
test_seq = og.initial_seq_no_stops(prot1_len, prot2_len, test_overlap, quiet=True)

# Warm up sequence generator
_ = og.overlapped_sequence_generator_int(
    DCA_params_1, DCA_params_2, test_seq,
    T1=MC_TEMP_1, T2=MC_TEMP_2,
    numberofiterations=1000,
    quiet=True,
    nat_mean1=mean_e1, nat_mean2=mean_e2,
    nat_std1=std_e1, nat_std2=std_e2,
    use_z_score=Z_SCORE
)

# Warm up is_path_stop_codon_free
test_arr = seq_to_array(test_seq)
test_positions = np.array([0, 1, 2], dtype=np.int32)
test_new_nts = np.array([0, 1, 2], dtype=np.uint8)
test_order = np.array([0, 1, 2], dtype=np.int32)
_ = is_path_stop_codon_free(test_order, test_arr, test_positions, test_new_nts, len_aa_1, len_aa_2)

print("JIT warmup complete!")

Warming up JIT compilation...
JIT warmup complete!


## 5. Sequence Generation Worker

In [8]:
def generate_sequence_worker(args):
    """
    Worker function for parallel sequence generation.
    
    Args:
        args: tuple of (overlap, seq_idx, seed, DCA_params_1, DCA_params_2, 
                       prot1_len, prot2_len, mc_iterations, mc_temp_1, mc_temp_2,
                       mean_e1, mean_e2, std_e1, std_e2, z_score)
    """
    (overlap, seq_idx, seed, Jvec1, hvec1, Jvec2, hvec2,
     prot1_len, prot2_len, mc_iterations, mc_temp_1, mc_temp_2,
     mean_e1, mean_e2, std_e1, std_e2, z_score) = args
    
    np.random.seed(seed)
    
    try:
        initial_seq = og.initial_seq_no_stops(prot1_len, prot2_len, overlap, quiet=True)
        
        result = og.overlapped_sequence_generator_int(
            (Jvec1, hvec1), (Jvec2, hvec2), initial_seq,
            T1=mc_temp_1, T2=mc_temp_2,
            numberofiterations=mc_iterations,
            quiet=True,
            whentosave=100.0,
            nat_mean1=mean_e1, nat_mean2=mean_e2,
            nat_std1=std_e1, nat_std2=std_e2,
            use_z_score=z_score
        )
        
        return {
            'overlap': overlap,
            'seq_idx': seq_idx,
            'success': True,
            'sequence': result[6],
            'energies': result[5]
        }
    except Exception as e:
        return {
            'overlap': overlap,
            'seq_idx': seq_idx,
            'success': False,
            'error': str(e)
        }

print("Sequence generation worker defined.")

Sequence generation worker defined.


## 6. Generate Functional Sequences (Parallel)

In [9]:
# Prepare work units for parallel sequence generation
work_units = []
base_seed = 42

for overlap in OVERLAPS:
    for seq_idx in range(N_SEQUENCES_PER_OVERLAP):
        seed = base_seed + overlap * 1000 + seq_idx
        work_units.append((
            overlap, seq_idx, seed,
            Jvec_1, hvec_1, Jvec_2, hvec_2,
            prot1_len, prot2_len,
            MC_ITERATIONS, MC_TEMP_1, MC_TEMP_2,
            mean_e1, mean_e2, std_e1, std_e2, Z_SCORE
        ))

print(f"Prepared {len(work_units)} sequence generation tasks")
print(f"({len(OVERLAPS)} overlaps x {N_SEQUENCES_PER_OVERLAP} sequences each)")

Prepared 1 sequence generation tasks
(1 overlaps x 1 sequences each)


In [ ]:
# Run parallel sequence generation
if __name__ == '__main__':
    print(f"Generating sequences using {N_WORKERS} workers...")
    start_time = time.time()
    
    with Pool(N_WORKERS) as pool:
        seq_results = list(tqdm(
            pool.imap(generate_sequence_worker, work_units),
            total=len(work_units),
            desc="Generating sequences"
        ))
    
    elapsed = time.time() - start_time
    
    # Organize results by overlap
    sequences_by_overlap = {}
    energies_by_overlap = {}
    
    for result in seq_results:
        if result['success']:
            overlap = result['overlap']
            if overlap not in sequences_by_overlap:
                sequences_by_overlap[overlap] = []
                energies_by_overlap[overlap] = []
            sequences_by_overlap[overlap].append(result['sequence'])
            energies_by_overlap[overlap].append(result['energies'])
    
    print(f"\nSequence generation complete in {elapsed:.1f}s")
    print(f"\nSequences generated per overlap:")
    for overlap in sorted(sequences_by_overlap.keys()):
        print(f"  Overlap {overlap}: {len(sequences_by_overlap[overlap])} sequences")

Generating sequences using 12 workers...


Generating sequences:   0%|          | 0/1 [00:00<?, ?it/s]

## 7. Run Stop-Codon-Free GA Optimization (with Convergence Tracking)

In [ ]:
def run_ga_with_convergence_tracking(seq_start, seq_end, 
                                      Jvec1, hvec1, Jvec2, hvec2,
                                      len_aa_1, len_aa_2,
                                      mean_e1, mean_e2, std_e1, std_e2, z_score,
                                      ga_pop, ga_gen, ga_mut_rate,
                                      max_init_retries, max_mutation_retries, max_crossover_retries,
                                      track_every_n=10):
    """
    Run GA with tracking of best path at regular intervals.
    Returns the convergence history showing how the best path evolves.
    """
    from ga_worker import (
        seq_to_array, get_mutations_arrays, get_mutation_path,
        is_path_stop_codon_free, evaluate_population_parallel,
        evaluate_path_fitness_numba, get_path_energies
    )
    
    # Setup
    seq_arr = seq_to_array(seq_start)
    mut_positions, mut_old_nts, mut_new_nts = get_mutations_arrays(seq_start, seq_end)
    mutations = get_mutation_path(seq_start, seq_end)
    n_mutations = len(mut_positions)
    
    if n_mutations == 0:
        return None
    
    nat_std_1 = std_e1 if std_e1 is not None else 1.0
    nat_std_2 = std_e2 if std_e2 is not None else 1.0
    
    def is_valid_path(path):
        return is_path_stop_codon_free(path, seq_arr, mut_positions, mut_new_nts, len_aa_1, len_aa_2)
    
    # Initialize population (stop-codon-free)
    population = np.zeros((ga_pop, n_mutations), dtype=np.int32)
    for i in range(ga_pop):
        for attempt in range(max_init_retries):
            candidate = np.random.permutation(n_mutations).astype(np.int32)
            if is_valid_path(candidate):
                population[i] = candidate
                break
        else:
            raise RuntimeError(f"Could not initialize population after {max_init_retries} attempts")
    
    # Track convergence
    convergence_history = []  # List of (generation, path_distances)
    elitism_count = max(1, int(0.1 * ga_pop))
    
    # Initial best
    best_individual = population[0].copy()
    best_fitness = evaluate_path_fitness_numba(
        best_individual, seq_arr, mut_positions, mut_new_nts,
        Jvec1, hvec1, Jvec2, hvec2, len_aa_1, len_aa_2,
        mean_e1, mean_e2, nat_std_1, nat_std_2, z_score
    )
    
    # Record initial best path (generation 0)
    _, initial_distances = get_path_energies(
        best_individual.tolist(), seq_start, mutations,
        Jvec1, hvec1, Jvec2, hvec2, len_aa_1, len_aa_2,
        mean_e1, mean_e2, nat_std_1=nat_std_1, nat_std_2=nat_std_2, z_score=z_score
    )
    convergence_history.append((0, initial_distances.tolist(), best_fitness))
    
    # GA loop
    for gen in range(ga_gen):
        # Evaluate population
        fitnesses = evaluate_population_parallel(
            population, seq_arr, mut_positions, mut_new_nts,
            Jvec1, hvec1, Jvec2, hvec2, len_aa_1, len_aa_2,
            mean_e1, mean_e2, nat_std_1, nat_std_2, z_score
        )
        
        # Track best
        gen_best_idx = np.argmin(fitnesses)
        if fitnesses[gen_best_idx] < best_fitness:
            best_fitness = fitnesses[gen_best_idx]
            best_individual = population[gen_best_idx].copy()
        
        # Record convergence every N generations
        if (gen + 1) % track_every_n == 0:
            _, path_distances = get_path_energies(
                best_individual.tolist(), seq_start, mutations,
                Jvec1, hvec1, Jvec2, hvec2, len_aa_1, len_aa_2,
                mean_e1, mean_e2, nat_std_1=nat_std_1, nat_std_2=nat_std_2, z_score=z_score
            )
            convergence_history.append((gen + 1, path_distances.tolist(), best_fitness))
        
        # Selection and reproduction
        sorted_idx = np.argsort(fitnesses)
        new_population = np.zeros_like(population)
        
        # Elitism
        for i in range(elitism_count):
            new_population[i] = population[sorted_idx[i]]
        
        # Crossover and mutation
        idx = elitism_count
        while idx < ga_pop:
            p1_idx = sorted_idx[np.random.choice(ga_pop // 2)]
            p2_idx = sorted_idx[np.random.choice(ga_pop // 2)]
            parent1, parent2 = population[p1_idx], population[p2_idx]
            
            # Crossover with stop-codon-free guarantee
            size = len(parent1)
            child = None
            for _ in range(max_crossover_retries):
                start, end = sorted(np.random.choice(size, 2, replace=False))
                candidate = np.full(size, -1, dtype=np.int32)
                candidate[start:end] = parent1[start:end]
                in_child = set(candidate[start:end])
                p2_remaining = [x for x in parent2 if x not in in_child]
                fill_idx = 0
                for i in range(size):
                    if candidate[i] == -1:
                        candidate[i] = p2_remaining[fill_idx]
                        fill_idx += 1
                if is_valid_path(candidate):
                    child = candidate
                    break
            if child is None:
                child = parent1.copy() if np.random.rand() < 0.5 else parent2.copy()
            
            # Mutation with stop-codon-free guarantee
            if np.random.rand() < ga_mut_rate and len(child) >= 2:
                original = child.copy()
                for _ in range(max_mutation_retries):
                    mutated = original.copy()
                    i, j = np.random.choice(len(mutated), 2, replace=False)
                    mutated[i], mutated[j] = mutated[j], mutated[i]
                    if is_valid_path(mutated):
                        child = mutated
                        break
            
            new_population[idx] = child
            idx += 1
        
        population = new_population
    
    # Final path energies
    final_energies, final_distances = get_path_energies(
        best_individual.tolist(), seq_start, mutations,
        Jvec1, hvec1, Jvec2, hvec2, len_aa_1, len_aa_2,
        mean_e1, mean_e2, nat_std_1=nat_std_1, nat_std_2=nat_std_2, z_score=z_score
    )
    
    return {
        'best_order': best_individual.tolist(),
        'best_fitness': best_fitness,
        'path_distances': final_distances.tolist(),
        'path_energies': final_energies.tolist(),
        'convergence_history': convergence_history,  # List of (gen, path_distances, fitness)
        'n_mutations': n_mutations
    }

print("GA with convergence tracking defined.")

In [ ]:
def run_ga_trial_with_tracking(args):
    """
    Worker function for parallel GA optimization with convergence tracking.
    """
    (overlap, trial_idx, seed, seq_start, seq_end,
     Jvec1, hvec1, Jvec2, hvec2, len_aa_1, len_aa_2,
     mean_e1, mean_e2, std_e1, std_e2, z_score,
     ga_pop, ga_gen, ga_mut_rate,
     max_init_retries, max_mutation_retries, max_crossover_retries,
     track_every_n) = args
    
    np.random.seed(seed)
    
    try:
        ham_dist = hamming_distance(seq_start, seq_end)
        
        result = run_ga_with_convergence_tracking(
            seq_start, seq_end,
            Jvec1, hvec1, Jvec2, hvec2, len_aa_1, len_aa_2,
            mean_e1, mean_e2, std_e1, std_e2, z_score,
            ga_pop, ga_gen, ga_mut_rate,
            max_init_retries, max_mutation_retries, max_crossover_retries,
            track_every_n
        )
        
        if result is None:
            return {
                'overlap': overlap,
                'trial': trial_idx,
                'success': False,
                'error': 'No mutations between sequences'
            }
        
        return {
            'overlap': overlap,
            'trial': trial_idx,
            'success': True,
            'hamming_distance': ham_dist,
            'n_mutations': result['n_mutations'],
            'max_distance': result['best_fitness'],
            'start_distance': result['path_distances'][0] if result['path_distances'] else np.nan,
            'end_distance': result['path_distances'][-1] if result['path_distances'] else np.nan,
            'best_order': result['best_order'],
            'path_distances': result['path_distances'],
            'convergence_history': result['convergence_history'],
            'seq_start': seq_start,
            'seq_end': seq_end,
            'ga_converged': result['best_fitness'] < result['convergence_history'][0][2]
        }
    except RuntimeError as e:
        return {
            'overlap': overlap,
            'trial': trial_idx,
            'success': False,
            'error': str(e)
        }
    except Exception as e:
        return {
            'overlap': overlap,
            'trial': trial_idx,
            'success': False,
            'error': str(e)
        }

print("GA trial worker with tracking defined.")

In [ ]:
# Prepare GA work units with convergence tracking
ga_work_units = []
ga_seed = 12345

for overlap in OVERLAPS:
    if overlap not in sequences_by_overlap or len(sequences_by_overlap[overlap]) < 2:
        print(f"Warning: Not enough sequences for overlap {overlap}")
        continue
    
    seqs = sequences_by_overlap[overlap]
    
    # Generate sequence pairs for trials
    for trial_idx in range(min(N_TRIALS_PER_OVERLAP, len(seqs) * (len(seqs) - 1) // 2)):
        # Select random pair of sequences
        i, j = np.random.choice(len(seqs), 2, replace=False)
        seq_start = seqs[i]
        seq_end = seqs[j]
        
        seed = ga_seed + overlap * 1000 + trial_idx
        
        ga_work_units.append((
            overlap, trial_idx, seed, seq_start, seq_end,
            Jvec_1, hvec_1, Jvec_2, hvec_2, len_aa_1, len_aa_2,
            mean_e1, mean_e2, std_e1, std_e2, Z_SCORE,
            GA_POPULATION, GA_GENERATIONS, GA_MUTATION_RATE,
            MAX_INIT_RETRIES, MAX_MUTATION_RETRIES, MAX_CROSSOVER_RETRIES,
            TRACK_EVERY_N_GENERATIONS
        ))

print(f"Prepared {len(ga_work_units)} GA optimization tasks")

In [ ]:
# Run parallel GA optimization with convergence tracking
if __name__ == '__main__':
    print(f"Running GA optimization using {N_WORKERS} workers...")
    print(f"(Stop-codon-free guarantee enabled, tracking every {TRACK_EVERY_N_GENERATIONS} generations)")
    start_time = time.time()
    
    with Pool(N_WORKERS) as pool:
        ga_results = list(tqdm(
            pool.imap(run_ga_trial_with_tracking, ga_work_units),
            total=len(ga_work_units),
            desc="GA Optimization"
        ))
    
    elapsed = time.time() - start_time
    
    # Count successes and failures
    successes = [r for r in ga_results if r['success']]
    failures = [r for r in ga_results if not r['success']]
    
    print(f"\nGA optimization complete in {elapsed:.1f}s")
    print(f"Successful trials: {len(successes)}")
    print(f"Failed trials: {len(failures)}")
    
    if failures:
        print("\nFailure reasons:")
        for f in failures[:5]:  # Show first 5
            print(f"  Overlap {f['overlap']}, Trial {f['trial']}: {f.get('error', 'Unknown')}")

## 8. Validation: Verify All Paths Are Stop-Codon Free

In [ ]:
# Verify all successful paths are actually stop-codon free
print("=" * 60)
print("VALIDATION: Checking all paths are stop-codon free")
print("=" * 60)

validation_passed = 0
validation_failed = 0

for result in tqdm(successes, desc="Validating paths"):
    seq_start = result['seq_start']
    seq_end = result['seq_end']
    best_order = np.array(result['best_order'], dtype=np.int32)
    
    # Get mutation arrays
    seq_arr = seq_to_array(seq_start)
    mut_positions, _, mut_new_nts = get_mutations_arrays(seq_start, seq_end)
    
    # Verify the path is stop-codon free
    is_valid = is_path_stop_codon_free(
        best_order, seq_arr, mut_positions, mut_new_nts,
        len_aa_1, len_aa_2
    )
    
    if is_valid:
        validation_passed += 1
    else:
        validation_failed += 1
        print(f"FAILED: Overlap {result['overlap']}, Trial {result['trial']}")

print(f"\nValidation Results:")
print(f"  Passed: {validation_passed}")
print(f"  Failed: {validation_failed}")

if validation_failed == 0:
    print("\n*** ALL PATHS VERIFIED STOP-CODON FREE! ***")
else:
    print(f"\n*** WARNING: {validation_failed} paths have stop codons! ***")

## 9. Results Analysis

In [ ]:
# Create results DataFrame
results_df = pd.DataFrame([{
    'overlap': r['overlap'],
    'trial': r['trial'],
    'reading_frame': r['overlap'] % 3,
    'hamming_distance': r['hamming_distance'],
    'n_mutations': r['n_mutations'],
    'max_distance': r['max_distance'],
    'start_distance': r['start_distance'],
    'end_distance': r['end_distance'],
    'ga_converged': r['ga_converged']
} for r in successes])

print("Results Summary:")
print(results_df.describe())

# Check for any infinite values (would indicate stop codons)
inf_count = np.isinf(results_df['max_distance']).sum()
print(f"\nPaths with infinite distance (stop codons): {inf_count}")

if inf_count == 0:
    print("*** No stop codons detected in any path! ***")

In [ ]:
# Summary by overlap
overlap_summary = results_df.groupby('overlap').agg({
    'max_distance': ['mean', 'std', 'min', 'max'],
    'hamming_distance': 'mean',
    'ga_converged': 'mean'
}).round(3)

print("Results by Overlap:")
print(overlap_summary)

## 10. GA Convergence Visualization

This plot shows how the GA optimizes the evolutionary path over generations.
- **X-axis**: Mutation step (# of mutations applied)
- **Y-axis**: Z-Score Distance from natural
- **Color**: Generation number (lighter = later generations = more optimized)

In [ ]:
# Plot GA Convergence: Mutation step vs Z-Score Distance, colored by generation

# Select a few representative trials to show convergence
sample_trials = []
for overlap in sorted(set(r['overlap'] for r in successes))[:4]:  # First 4 overlaps
    overlap_results = [r for r in successes if r['overlap'] == overlap]
    if overlap_results:
        sample_trials.append(overlap_results[0])

n_samples = len(sample_trials)
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for idx, result in enumerate(sample_trials):
    if idx >= 4:
        break
    
    ax = axes[idx]
    convergence = result['convergence_history']
    n_mutations = result['n_mutations']
    
    # Color map: early generations are dark, later generations are light
    generations = [c[0] for c in convergence]
    max_gen = max(generations)
    cmap = cm.plasma
    norm = Normalize(vmin=0, vmax=max_gen)
    
    # Plot each generation's best path
    for gen, path_distances, fitness in convergence:
        color = cmap(norm(gen))
        x = np.arange(len(path_distances))
        ax.plot(x, path_distances, '-', color=color, linewidth=1.5, alpha=0.7)
    
    # Add colorbar
    sm = cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax)
    cbar.set_label('Generation', fontsize=10)
    
    # Highlight final path
    final_path = result['path_distances']
    ax.plot(np.arange(len(final_path)), final_path, 'k-', linewidth=2.5, 
            label=f'Final (Gen {GA_GENERATIONS})', zorder=10)
    
    # Mark start and end
    ax.scatter([0], [final_path[0]], s=100, c='green', zorder=15, 
               label='Start', edgecolors='black', linewidth=1)
    ax.scatter([len(final_path)-1], [final_path[-1]], s=100, c='red', zorder=15, 
               label='End', edgecolors='black', linewidth=1)
    
    # Labels
    ax.set_xlabel('# of Mutations', fontsize=11)
    ax.set_ylabel(DIST_LABEL, fontsize=11)
    ax.set_title(f'Overlap = {result["overlap"]} nt (Frame {result["overlap"] % 3})\n'
                 f'Max: {result["max_distance"]:.3f}', fontsize=11)
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(True, alpha=0.3)

# Hide unused axes
for j in range(len(sample_trials), len(axes)):
    axes[j].axis('off')

plt.suptitle(f'GA Convergence: How the Best Path Evolves\n'
             f'(Tracking every {TRACK_EVERY_N_GENERATIONS} generations, darker = earlier)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('ga_convergence_paths.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: ga_convergence_paths.png")

In [ ]:
# Single detailed convergence plot for the first successful trial
if successes:
    result = successes[0]
    convergence = result['convergence_history']
    
    fig, ax = plt.subplots(figsize=(14, 8))
    
    # Color map
    generations = [c[0] for c in convergence]
    max_gen = max(generations)
    cmap = cm.viridis
    norm = Normalize(vmin=0, vmax=max_gen)
    
    # Plot each generation's best path with increasing opacity
    for i, (gen, path_distances, fitness) in enumerate(convergence):
        color = cmap(norm(gen))
        alpha = 0.3 + 0.7 * (i / len(convergence))  # Increasing opacity
        x = np.arange(len(path_distances))
        label = f'Gen {gen} (max={fitness:.3f})' if gen % (TRACK_EVERY_N_GENERATIONS * 3) == 0 or gen == 0 else None
        ax.plot(x, path_distances, '-', color=color, linewidth=1.5, alpha=alpha, label=label)
    
    # Highlight final path
    final_path = result['path_distances']
    ax.plot(np.arange(len(final_path)), final_path, 'r-', linewidth=3, 
            label=f'Final Path (max={result["max_distance"]:.3f})', zorder=20)
    
    # Add colorbar
    sm = cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, pad=0.02)
    cbar.set_label('Generation', fontsize=12)
    
    # Labels
    ax.set_xlabel('# of Mutations Applied', fontsize=12)
    ax.set_ylabel(DIST_LABEL, fontsize=12)
    ax.set_title(f'GA Convergence: Mutation Path Optimization\n'
                 f'Overlap = {result["overlap"]} nt, {result["n_mutations"]} mutations\n'
                 f'Dark = Early Generations, Light = Later Generations',
                 fontsize=13, fontweight='bold')
    ax.legend(loc='upper right', fontsize=9, framealpha=0.9)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('ga_convergence_detail.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: ga_convergence_detail.png")

In [ ]:
# Plot fitness (max distance) improvement over generations
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Fitness curves for all trials
ax1 = axes[0]
cmap = cm.viridis
overlaps_unique = sorted(set(r['overlap'] for r in successes))
norm = Normalize(vmin=min(overlaps_unique), vmax=max(overlaps_unique))

for result in successes:
    convergence = result['convergence_history']
    gens = [c[0] for c in convergence]
    fitnesses = [c[2] for c in convergence]
    color = cmap(norm(result['overlap']))
    ax1.plot(gens, fitnesses, '-', color=color, alpha=0.6, linewidth=1.5)

sm = cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax1)
cbar.set_label('Overlap (nt)', fontsize=10)

ax1.set_xlabel('Generation', fontsize=11)
ax1.set_ylabel(f'Max {DIST_LABEL} (Fitness)', fontsize=11)
ax1.set_title('GA Fitness Improvement Over Generations', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3)

# Right: Average improvement by generation
ax2 = axes[1]
all_gens = set()
for result in successes:
    for gen, _, _ in result['convergence_history']:
        all_gens.add(gen)
all_gens = sorted(all_gens)

mean_fitness_by_gen = []
std_fitness_by_gen = []
for gen in all_gens:
    fitnesses_at_gen = []
    for result in successes:
        for g, _, f in result['convergence_history']:
            if g == gen:
                fitnesses_at_gen.append(f)
                break
    if fitnesses_at_gen:
        mean_fitness_by_gen.append(np.mean(fitnesses_at_gen))
        std_fitness_by_gen.append(np.std(fitnesses_at_gen))

mean_fitness_by_gen = np.array(mean_fitness_by_gen)
std_fitness_by_gen = np.array(std_fitness_by_gen)

ax2.plot(all_gens, mean_fitness_by_gen, 'b-', linewidth=2, label='Mean')
ax2.fill_between(all_gens, 
                  mean_fitness_by_gen - std_fitness_by_gen,
                  mean_fitness_by_gen + std_fitness_by_gen,
                  alpha=0.3, color='blue', label='Std Dev')

ax2.set_xlabel('Generation', fontsize=11)
ax2.set_ylabel(f'Max {DIST_LABEL} (Fitness)', fontsize=11)
ax2.set_title('Average GA Convergence Across All Trials', fontsize=12, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('ga_fitness_convergence.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: ga_fitness_convergence.png")

## 11. Additional Visualizations

In [ ]:
# Plot 1: Max distance by overlap
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot by overlap
sns.boxplot(data=results_df, x='overlap', y='max_distance', ax=axes[0], palette='viridis')
axes[0].set_xlabel('Overlap (nucleotides)')
axes[0].set_ylabel(f'Max {DIST_LABEL}')
axes[0].set_title('Energy Barrier by Overlap (Stop-Codon-Free GA)')

# Box plot by reading frame
frame_labels = {0: 'Frame 0\n(in-frame)', 1: 'Frame +1', 2: 'Frame +2'}
results_df['frame_label'] = results_df['reading_frame'].map(frame_labels)
sns.boxplot(data=results_df, x='frame_label', y='max_distance', ax=axes[1], palette='Set2')
axes[1].set_xlabel('Reading Frame')
axes[1].set_ylabel(f'Max {DIST_LABEL}')
axes[1].set_title('Energy Barrier by Reading Frame')

plt.tight_layout()
plt.savefig('stop_codon_free_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: stop_codon_free_results.png")

In [ ]:
# Plot: Correlation between Hamming distance and max energy barrier
fig, ax = plt.subplots(figsize=(10, 7))

ham_dists = results_df['hamming_distance'].values
max_dists = results_df['max_distance'].values

# Scatter plot colored by reading frame
frame_colors = {0: '#66c2a5', 1: '#fc8d62', 2: '#8da0cb'}
for frame in [0, 1, 2]:
    mask = results_df['reading_frame'] == frame
    ax.scatter(ham_dists[mask], max_dists[mask], 
               s=100, c=frame_colors[frame], alpha=0.7,
               label=f'Frame {frame}', edgecolors='white', linewidth=0.5)

# Add regression line
z = np.polyfit(ham_dists, max_dists, 1)
p = np.poly1d(z)
x_line = np.linspace(ham_dists.min(), ham_dists.max(), 100)
ax.plot(x_line, p(x_line), 'k--', linewidth=2, alpha=0.7)

# Calculate correlation
corr, p_value = stats.pearsonr(ham_dists, max_dists)
ax.text(0.05, 0.95, f'r = {corr:.3f}\np = {p_value:.2e}', 
        transform=ax.transAxes, fontsize=12, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

ax.set_xlabel('Hamming Distance', fontsize=12)
ax.set_ylabel(f'Max {DIST_LABEL}', fontsize=12)
ax.set_title(f'Hamming Distance vs Energy Barrier\n(Stop-Codon-Free GA, {PROTEIN_1} + {PROTEIN_2})', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('stop_codon_free_hamming_vs_barrier.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: stop_codon_free_hamming_vs_barrier.png")

## 12. Save Results

In [ ]:
# Save results to CSV
results_df['protein_1'] = PROTEIN_1
results_df['protein_2'] = PROTEIN_2
results_df.to_csv('stop_codon_free_ga_results.csv', index=False)
print("Results saved to: stop_codon_free_ga_results.csv")

# Summary
print("\n" + "=" * 60)
print("STOP-CODON-FREE GA ANALYSIS COMPLETE")
print("=" * 60)
print(f"Total successful trials: {len(successes)}")
print(f"Paths with stop codons: 0 (verified)")
print(f"Mean max distance: {results_df['max_distance'].mean():.3f}")
print(f"Std max distance: {results_df['max_distance'].std():.3f}")
print("=" * 60)